# FLAG 2027 — NB-87: bridge, FACELORA-03 frames + B3 checkpoint -> private dataset (CPU)

In [ ]:
print("[boot] FLAG_87_bridge | python", __import__("sys").version.split()[0], "| start", __import__("time").strftime("%Y-%m-%d %H:%M:%S"), flush=True)  # heartbeat rule: first output, before any import
import os, sys, json, time, shutil
from pathlib import Path
IN = Path("/kaggle/input")
HANDLE = f"onnhqunhh/flag2027-lora-frames-p"
log = lambda *a: print(time.strftime("%H:%M:%S"), *a, flush=True)
import kagglehub
log("kagglehub", kagglehub.__version__)

In [ ]:
# stage = symlinks with the published names (no copy); every source must be found once
def find(name, parent=None):
    hits = sorted(p for p in IN.rglob(name) if parent is None or p.parent.name == parent)
    assert hits, f"{name} ({parent}) not found: attach the FLAG_45 / FLAG_63 / FLAG_65-b outputs"
    return hits[0]


stage = Path("/tmp/stage"); shutil.rmtree(stage, ignore_errors=True)
z37, man = find("vox_frames_png.zip"), find("frames_manifest.json")
ck = find("facelora_ckpt.pt", "feats_facelora3_b")
links = {stage / "vox_frames_png.zipdata": z37, stage / "frames/frames_manifest.json": man,
         stage / "feats_facelora3_b/facelora_ckpt.pt": ck}
for part in ("vb", "vox2b"):
    for zf in sorted((man.parent / part).glob("*.zip")):
        links[stage / f"frames/{part}/{zf.stem}.zipdata"] = zf
for dst, src in links.items():
    dst.parent.mkdir(parents=True, exist_ok=True); os.symlink(src, dst)
m = json.loads(man.read_text())
n = {p: len(list((stage / f"frames/{p}").glob("*.zipdata"))) for p in ("vb", "vox2b")}
gb = sum(Path(s).stat().st_size for s in links.values()) / 2 ** 30
log(f"staged {len(links)} files, {gb:.2f} GB | chunk zips {n} vs manifest {{'vb': {m['parts']['vb']['zips']}, 'vox2b': {m['parts']['vox2b']['zips']}}}")
assert n["vb"] == m["parts"]["vb"]["zips"] and n["vox2b"] == m["parts"]["vox2b"]["zips"], "missing frame zips"

In [ ]:
# upload as a NEW private dataset of this account (kagglehub: private by default); privatize locally afterwards
t0 = time.time()
kagglehub.dataset_upload(HANDLE, str(stage), version_notes="FACELORA-03 inputs: FLAG_45 + FLAG_63 frames, FLAG_65-b checkpoint (14/10)")
log(f"uploaded to {HANDLE} [{time.time() - t0:.0f}s]")